In [6]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils import resample
import joblib
import spacy
nlp = spacy.load("en_core_web_sm")

In [7]:

def extract_claim_source(text):
    doc = nlp(text)

    sources = {
        "ORG": [],
        "PERSON": [],
        "GPE": []
    }

    for ent in doc.ents:
        if ent.label_ in sources:
            sources[ent.label_].append(ent.text)

    if sources["ORG"]:
        return f"Organization claim: {sources['ORG'][0]}"
    elif sources["PERSON"]:
        return f"Person claim: {sources['PERSON'][0]}"
    elif sources["GPE"]:
        return f"Location based claim: {sources['GPE'][0]}"
    else:
        return "No clear source found"

In [8]:
def load_dataset(fake_path="Fake.csv", true_path="True.csv"):
    fake_df = pd.read_csv(fake_path)
    true_df = pd.read_csv(true_path)
    
    fake_df['label'] = 0  # Fake news = 0
    true_df['label'] = 1  # Real news = 1

    df = pd.concat([fake_df, true_df], ignore_index=True)
    return df

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'\W', ' ', text)      # Remove special chars
    text = re.sub(r'\d+', ' ', text)     # Remove digits
    text = re.sub(r'\s+', ' ', text)     # Remove extra spaces
    return text.strip()

In [10]:
def clean_dataset(df):
    df['text'] = df['text'].apply(clean_text)
    return df

In [11]:
def balance_dataset(df):
    df_fake = df[df['label'] == 0]
    df_real = df[df['label'] == 1]

    df_real_upsampled = resample(
        df_real,
        replace=True,
        n_samples=len(df_fake),
        random_state=42
    )

    df_balanced = pd.concat([df_fake, df_real_upsampled])
    df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)
    return df_balanced

In [12]:
def train_model(df):
    vectorizer = TfidfVectorizer(
        stop_words='english',
        max_df=0.7,
        min_df=2,
        ngram_range=(1,2)
    )

    X = vectorizer.fit_transform(df['text'])
    y = df['label']

    model = MultinomialNB()
    model.fit(X, y)

    joblib.dump(model, "fake_news_model.pkl")
    joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

    return model, vectorizer

In [13]:
def load_model():
    model = joblib.load("fake_news_model.pkl")
    vectorizer = joblib.load("tfidf_vectorizer.pkl")
    return model, vectorizer

In [14]:
def predict_single(news_text, model, vectorizer):
    cleaned = clean_text(news_text)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    confidence = max(prob) * 100
    label = "REAL NEWS" if prediction == 1 else "FAKE NEWS"
    return label, confidence

In [15]:
# def predict_batch(csv_path, model, vectorizer):
#     df = pd.read_csv(csv_path)
#     if 'text' not in df.columns:
#         raise ValueError("CSV must have a column named 'text'")
#     df['cleaned_text'] = df['text'].apply(clean_text)
#     X_batch = vectorizer.transform(df['cleaned_text'])
#     df['prediction'] = model.predict(X_batch)
#     df['confidence'] = model.predict_proba(X_batch).max(axis=1)*100
#     df['prediction_label'] = df['prediction'].apply(lambda x: "REAL" if x==1 else "FAKE")
#     return df[['text','prediction_label','confidence']]

def predict_single(news_text, model, vectorizer):
    cleaned = clean_text(news_text)
    vec = vectorizer.transform([cleaned])
    prediction = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    confidence = max(prob) * 100
    label = "REAL NEWS" if prediction == 1 else "FAKE NEWS"

    source = extract_claim_source(news_text)

    return label, confidence, source

In [17]:
if __name__ == "__main__":
    # 1️⃣ Load and clean
    df = load_dataset()
    df = clean_dataset(df)
    df = balance_dataset(df)

    # 2️⃣ Train model
    model, vectorizer = train_model(df)

    # 3️⃣ Test prediction
    test_news = "The government launched a new digital education policy."
    label, confidence,source= predict_single(test_news, model, vectorizer)
    print(f"Prediction: {label}, Confidence: {confidence:.2f}%,Source: {source}")

Prediction: REAL NEWS, Confidence: 72.58%,Source: No clear source found
